# Session 8 (lakehouse): The Trust Test, from the lake

**Goal:** answer session 1's eight Trust Test questions with SQL over the home-lab lakehouse, and check that the lake gives exactly the answers the class got after session 4.

The first seven sessions cleanse a CSV with pandas. The lakehouse runs the same course code every morning and keeps the result as tables: **bronze** (the file as written), **silver** (cleansed and typed by the course's own sessions 2 to 4), and **gold** (ready for reporting). Here you query them directly.

> **Mock data only.** Everything here is fictional, and this connection is read-only: nothing you run can change the lake.

## 1. Connect

One line connects SQL cells to the lake, read-only. After it, a cell that starts with `%%sql` is a SQL query.

In [ ]:
%load_ext lakehouse.sql_magic

## 2. Look around

Each layer is a schema. Count the tables in each:

In [ ]:
%%sql
SELECT table_schema, count(*) AS tables
FROM information_schema.tables
WHERE table_catalog = 'lake'
GROUP BY 1
ORDER BY 1

Bronze keeps the file exactly as written, every column as text. The brackets show the spaces and tabs that fooled session 1:

In [ ]:
%%sql
SELECT '[' || coalesce(gender, '') || ']' AS gender_as_written, count(*) AS records
FROM bronze.hero_offenders_raw
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10

## 3. The eight questions, from silver

Silver is the data after sessions 2 to 4, so each answer here should be the trustworthy one.

**Q1. Records missing a custody level.** Session 1's default load said 1,536. Whitespace-only values hid the rest; silver turned them into real nulls.

In [ ]:
%%sql
SELECT count(*) FILTER (WHERE custody_level IS NULL) AS missing_custody_level
FROM silver.hero_offenders

**Q2. Distinct gender values.** Session 1 counted 19 spellings. After stripping and the data owner's codes, there are two values plus missing.

In [ ]:
%%sql
SELECT coalesce(gender, '(missing)') AS gender, count(*) AS records
FROM silver.hero_offenders
GROUP BY 1
ORDER BY 2 DESC

**Q3. Records with last name KENT.** Exact text found none: names arrive in random casing. Session 2 matches on a standardized name, `last_name_std`.

In [ ]:
%%sql
SELECT count(*) AS kent_records
FROM silver.hero_offenders
WHERE last_name_std = 'KENT'

**Q4. Earliest projected release date.** Sorted as text, the "earliest" date was a tab and a space. Session 3 parsed nine formats into real dates.

In [ ]:
%%sql
SELECT min(projected_release_date) AS earliest_release
FROM silver.hero_offenders

**Q5. Does offender #02381457 exist?.** A text lookup missed it. Session 2 checked every ID and stored it as a whole number, so look it up as a number.

In [ ]:
%%sql
SELECT offender_number, lpad(offender_number::VARCHAR, 8, '0') AS shown_to_people
FROM silver.hero_offenders
WHERE offender_number = 2381457

**Q6. Average age at sentencing.** Impossible with dates as text. Session 3's flagged birth dates (Wonder Woman, 1213) are left out of the average.

In [ ]:
%%sql
SELECT round(avg(age_at_sentencing), 1) AS average_age
FROM silver.hero_offenders
WHERE dob_flag IS NULL

**Q7. Records flagged as escape risk.** Counting only "Y" missed Yes, y and the T/F system. Session 4 mapped every spelling to a real boolean.

In [ ]:
%%sql
SELECT count(*) FILTER (WHERE escape_risk) AS escape_risk
FROM silver.hero_offenders

**Q8. Records in protective custody.** The same single-character Y/N and T/F trap, mapped explicitly in session 4.

In [ ]:
%%sql
SELECT count(*) FILTER (WHERE protective_custody) AS protective_custody
FROM silver.hero_offenders

## 4. Check against the course

The course's Trust Test scorecard is in `gold.trust_test`, computed by the course's own Python every morning. Your SQL should agree with every answer.

In [ ]:
# The same eight answers in one query, as text, compared with the course's own scorecard (gold.trust_test).
mine_rows = %sql SELECT 'Q1', (count(*) FILTER (WHERE custody_level IS NULL))::VARCHAR FROM silver.hero_offenders UNION ALL SELECT 'Q2', (count(DISTINCT gender) + max((gender IS NULL)::INTEGER))::VARCHAR FROM silver.hero_offenders UNION ALL SELECT 'Q3', (count(*) FILTER (WHERE last_name_std = 'KENT'))::VARCHAR FROM silver.hero_offenders UNION ALL SELECT 'Q4', min(projected_release_date)::VARCHAR FROM silver.hero_offenders UNION ALL SELECT 'Q5', CASE WHEN bool_or(offender_number = 2381457) THEN 'True' ELSE 'False' END FROM silver.hero_offenders UNION ALL SELECT 'Q6', round(avg(age_at_sentencing) FILTER (WHERE dob_flag IS NULL), 1)::VARCHAR FROM silver.hero_offenders UNION ALL SELECT 'Q7', (count(*) FILTER (WHERE escape_risk))::VARCHAR FROM silver.hero_offenders UNION ALL SELECT 'Q8', (count(*) FILTER (WHERE protective_custody))::VARCHAR FROM silver.hero_offenders
course_rows = %sql SELECT question_id, cleansed FROM gold.trust_test
mine, course = dict(list(mine_rows)), dict(list(course_rows))   # question -> answer

for q in sorted(course):
    print(f"{q}  lake: {mine[q]:>12}   course: {course[q]:>12}   {'match' if mine[q] == course[q] else 'DIFFERENT'}")
matches = sum(mine[q] == course[q] for q in course)
print(f"\n{matches} of {len(course)} answers match gold.trust_test")
assert matches == len(course), "the lake and the course disagree: check the latest daily_lake run"

## Wrap-up

- The lake holds the same answers the class reached in session 4, because silver is built by the course's own code.
- Bronze keeps the mess as written, so you can always show where an answer came from.
- SQL is one more way in; the rules stay the same.